<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/Quantum_Chromodynamics_Animation_Feynman_Diagrams.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quantum Chromodynamics (QCD) Scientific Animation

This notebook generates a high-fidelity scientific animation visualizing the fundamental interactions of the Strong Nuclear Force as described by **Quantum Chromodynamics (QCD)**.

## Visualized Concepts
1.  **Quark–Gluon Vertex**: Shows a quark changing color charge by emitting a gluon.
2.  **Three-Gluon Vertex**: Illustrates gluon self-interaction, a unique feature of QCD.
3.  **Gluon Splitting**: A virtual gluon fluctuating into a quark–antiquark pair.
4.  **Color Confinement**: Demonstrates how pulling quarks apart increases energy until the 'string' breaks, creating new particles.
5.  **Asymptotic Freedom**: Visualizes how the strong force weakens at extremely short distances.

## Technical Specifications
- **Framework**: Matplotlib Animation API
- **Resolution**: 1280x720 (720p)
- **Frame Rate**: 30 FPS
- **Duration**: ~61 Seconds
- **Output Format**: MP4 (H.264)

## Attribution
- **Author**: Mugambi Ndwiga
- **Socials**: [@craftsandengineering](https://www.instagram.com/craftsandengineering)
- **License**: Creative Commons Attribution 4.0 International


In [9]:
"""
Quantum Chromodynamics — Scientific Animation

Author:
Mugambi Ndwiga

Instagram:
@craftsandengineering

Description:
Educational scientific animation visualizing color charge,
gluons, confinement, asymptotic freedom, and proton structure
using Quantum Chromodynamics (QCD).

Platform:
Google Colab

Libraries:
matplotlib, numpy
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.animation as animation
from IPython.display import HTML
from base64 import b64encode
import google.colab.files

# --- CONFIGURATION ---
FPS, DURATION = 30, 61
TOTAL_FRAMES = DURATION * FPS
INTRO_DURATION = 1 * FPS
SCENE_DURATION = 9 * FPS
BG_COLOR = '#080810'
COLOR_RED, COLOR_GREEN, COLOR_BLUE = '#ff5252', '#69f0ae', '#448aff'
COLOR_ANTI_RED, COLOR_ANTI_GREEN, COLOR_ANTI_BLUE = '#00FFAB', '#FF52FF', '#FFDF00'
TEXT_COLOR, HADRON_BLOB = '#f5f5f5', '#1a1a2e'

# --- HELPERS ---
def ease_in_out(t): return 0.5 - 0.5 * np.cos(np.pi * t)
def lerp(a, b, t): return a + (b - a) * t
def get_zigzag_line(p1, p2, amp=0.12, wavelen=0.3, phase=0):
    p1, p2 = np.array(p1), np.array(p2)
    dist = np.linalg.norm(p2-p1)
    t = np.linspace(0, 1, max(2, int(dist * 50)))
    base = p1 + np.outer(t, (p2-p1))
    perp = np.array([-(p2[1]-p1[1]), p2[0]-p1[0]]) / (dist + 1e-9)
    offsets = amp * np.sign(np.sin(2 * np.pi * dist * t / wavelen + phase))
    return base + offsets[:, np.newaxis] * perp

# --- ENGINE ---
fig, ax = plt.subplots(figsize=(12.8, 7.2), dpi=100)
fig.patch.set_facecolor(BG_COLOR)
ax.set_facecolor(BG_COLOR)
ax.set_xlim(-4.5, 4.5); ax.set_ylim(-3.5, 3.5); ax.axis('off')

watermark = ax.text(4.3, -3.3, 'Mugambi Ndwiga | @craftsandengineering', color=TEXT_COLOR, alpha=0.3, fontsize=10, ha='right')
title_text = ax.text(0, 2.8, '', color=TEXT_COLOR, fontsize=22, ha='center', fontweight='bold', alpha=0)
bottom_text = ax.text(0, -2.8, '', color=TEXT_COLOR, fontsize=15, ha='center', alpha=0)
sec_text = ax.text(0, -3.2, '', color=TEXT_COLOR, fontsize=12, ha='center', alpha=0)

gluon_artists = [ax.plot([], [], lw=1.5, alpha=0)[0] for _ in range(15)]
quark_artists = [ax.plot([], [], 'o', ms=10, alpha=0)[0] for _ in range(15)]
flux_tube = ax.add_patch(patches.Polygon([[0,0]], closed=True, color=COLOR_RED, alpha=0))

def update(f):
    if f < INTRO_DURATION:
        si, st = 0, f / INTRO_DURATION
    else:
        f_adj = f - INTRO_DURATION
        si, st = (f_adj // SCENE_DURATION) + 1, (f_adj % SCENE_DURATION) / SCENE_DURATION

    alpha = ease_in_out(min(1, st*10)) if st < 0.9 else ease_in_out(max(0, (1-st)*10))
    for a in gluon_artists + quark_artists: a.set_data([], []); a.set_alpha(0)
    flux_tube.set_alpha(0); sec_text.set_text('')

    if si == 0:
        title_text.set_text('The Strong Force')
        bottom_text.set_text('Quantum Chromodynamics and Color Charge')
    elif si == 1:
        title_text.set_text('Quark–Gluon Vertex')
        bottom_text.set_text('A quark changes color by emitting a gluon')
        q_pos = [lerp(-2, 0, st*2), 0] if st < 0.5 else [lerp(0, 2, (st-0.5)*2), lerp(0, -1, (st-0.5)*2)]
        quark_artists[0].set_data([q_pos[0]], [q_pos[1]]); quark_artists[0].set_color(COLOR_RED if st < 0.5 else COLOR_GREEN); quark_artists[0].set_alpha(1)
        if st > 0.5:
            gz = get_zigzag_line([0,0], [1.5, 1.5], phase=f*0.5)
            gluon_artists[0].set_data(gz[:,0], gz[:,1]); gluon_artists[0].set_color(COLOR_RED); gluon_artists[0].set_alpha(1)
    elif si == 2:
        title_text.set_text('Three-Gluon Vertex')
        bottom_text.set_text('Gluons carry color charge and self-interact')
        for i, ang in enumerate([0, 120, 240]):
            p = [np.cos(np.radians(ang))*2*(1-st), np.sin(np.radians(ang))*2*(1-st)]
            gz = get_zigzag_line(p, [0,0], phase=f*0.5)
            gluon_artists[i].set_data(gz[:,0], gz[:,1]); gluon_artists[i].set_alpha(1)
    elif si == 3:
        title_text.set_text('Gluon Splitting')
        bottom_text.set_text('A gluon fluctuates into a quark–antiquark pair')
        gz = get_zigzag_line([-2.5, 0], [0, 0], phase=f*0.5)
        gluon_artists[0].set_data(gz[:,0], gz[:,1]); gluon_artists[0].set_alpha(1 if st < 0.5 else (1-st)*2)
        if st > 0.5:
            ext = (st-0.5)*4
            quark_artists[0].set_data([ext], [ext*0.5]); quark_artists[0].set_color(COLOR_RED); quark_artists[0].set_alpha(1)
            quark_artists[1].set_data([ext], [-ext*0.5]); quark_artists[1].set_color(COLOR_ANTI_RED); quark_artists[1].set_alpha(1)
    elif si == 4:
        title_text.set_text('Color Confinement')
        bottom_text.set_text('Pulling quarks apart creates new quark pairs')
        dist = lerp(0.5, 3.5, st)
        if st < 0.7:
            quark_artists[0].set_data([-dist], [0]); quark_artists[0].set_color(COLOR_RED); quark_artists[0].set_alpha(1)
            quark_artists[1].set_data([dist], [0]); quark_artists[1].set_color(COLOR_ANTI_RED); quark_artists[1].set_alpha(1)
            flux_tube.set_xy([[-dist, 0.1], [dist, 0.1], [dist, -0.1], [-dist, -0.1]]); flux_tube.set_alpha(0.4)
        else:
            title_text.set_text('String Breaking')
            for i, x in enumerate([-dist, -dist+1, dist-1, dist]):
                quark_artists[i].set_data([x], [0]); quark_artists[i].set_alpha(1)
                quark_artists[i].set_color([COLOR_RED, COLOR_ANTI_RED, COLOR_RED, COLOR_ANTI_RED][i])
    elif si == 5:
        title_text.set_text('Asymptotic Freedom')
        bottom_text.set_text('The strong force weakens at short distances')
        for i in range(3):
            ang = np.radians(i*120 + f*2)
            r = 0.5 + (1-st)*1.5
            quark_artists[i].set_data([np.cos(ang)*r], [np.sin(ang)*r])
            quark_artists[i].set_color([COLOR_RED, COLOR_GREEN, COLOR_BLUE][i]); quark_artists[i].set_alpha(1)
    elif si >= 6:
        title_text.set_text('Created by Mugambi Ndwiga')
        bottom_text.set_text('@craftsandengineering\nScientific Animation with Matplotlib')

    title_text.set_alpha(alpha); bottom_text.set_alpha(alpha); sec_text.set_alpha(alpha)
    return [title_text, bottom_text, watermark, sec_text, flux_tube] + gluon_artists + quark_artists

ani = animation.FuncAnimation(fig, update, frames=TOTAL_FRAMES, interval=1000/FPS, blit=True)
ani.save('qcd_color_force.mp4', writer='ffmpeg', fps=FPS, bitrate=3000)
plt.close()

mp4 = open('qcd_color_force.mp4','rb').read()
data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()
display(HTML(f'<video width=960 controls autoplay loop><source src="{data_url}" type="video/mp4"></video>'))
google.colab.files.download('qcd_color_force.mp4')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>